# Fine tune AG-NEWS Classification Model to IMDB Dataset With LORA

In [29]:
import torch
import math
import torch.nn as nn
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

#### **Dataset Loading**

In [17]:
dataset = load_dataset("AG_NEWS")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [18]:
dataset['train'][0]

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'label': 2}

In [19]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
tokenizer

BertTokenizer(name_or_path='bert-base-cased', vocab_size=28996, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

#### **Define Tokenizer Function**

In [20]:
def tokenize_text (data):
    return tokenizer(data['text'], padding='max_length', truncation=True)


In [21]:
dataset = dataset.map(tokenize_text, batched=True)


In [22]:
dataset.column_names

{'train': ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
 'test': ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']}

In [23]:
dataset = dataset.rename_column('label', 'labels')
dataset.set_format('torch')

In [24]:
dataset_train = dataset['train']
dataset_test = dataset['test']

In [25]:
dataset_train[0]

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'labels': tensor(2),
 'input_ids': tensor([  101,  6250,  1457,   119, 10169,   140,  9598,  4388, 14000,  1103,
          2117,   113, 11336, 27603,   114, 11336, 27603,   118,  6373,   118,
         18275,  1116,   117,  6250,  1715,   112,   188,   173, 11129,  1979,
           165,  1467,  1104, 18737,   118,   172,  5730,  4724,   117,  1132,
          3195,  2448,  1254,   119,   102,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,  

**Split train set**

In [26]:
train_split, valid_split = dataset_train.train_test_split(0.1, seed=42)

### **Define Dataloaders**

In [27]:
BATCH_SIZE = 64

In [28]:
train_dataloader = DataLoader(train_split, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
valid_dataloader = DataLoader(valid_split, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_dataloader = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

### **Define Text-Embedding**

In [30]:
class TextEmbeddings(nn.Module):
    
    def __init__(self, vocab_size, embed_dims, dropout):
        super().__init__()
        
        self.embed_dim = embed_dims
        self.dropout = nn.Dropout(dropout)
        self.embeddings = nn.Embedding(vocab_size, embed_dims)
    
    
    def forward (self, input_tokens):
        # input_token -> [batch_size, seq_len]
        embed_out = self.embeddings(input_tokens) * math.sqrt(self.embed_dim)
        return self.dropout(embed_out)

### **Positional Embeddings**

In [31]:
class PositionalEmbedding (nn.Module):
    
    def __init__(self, vocab_size, d_model, max_seq_len, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        positions = torch.arange(0, max_seq_len).unsqueeze(1)
        
        pe = torch.zeros(max_seq_len, d_model).float()
        
        div_term = torch.exp(
            torch.arange(0,d_model,2) * (-math.log(10000)/d_model)
        )
        
        pe[:,0::2] = torch.sin(positions * div_term)
        pe[:,1::2] = torch.cos(positions * div_term)
        
        pe = pe.unsqueeze(0)
        
        self.register_buffer("pe",pe)
        
    def forward(self, text_embeddings):
        
        text_embed_size = text_embeddings.size(1)
        pos_embed = text_embeddings + self.pe[:,0:text_embed_size,:]
        return self.dropout(pos_embed)
        